<a href="https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/finite_element_method/solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solution

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/finite_element_method/solution.ipynb)

We now arrive at a matrix equation for the unknowns $u_i$, which is merely a system of equations as we've spoken about beforehand.

### Linear PDE

If the PDE is linear, we can solve it with a linear solver which is sparse due to the small support requirement of the elements. The matrix will not in general be banded however, due to the irregular connectivity of the mesh vertices.

In [1]:
try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## Example: 1D heat equation

Solve the 1D heat balance equation:

$$ \frac{\partial T}{\partial t} = \nabla \cdot \lambda \nabla T$$

with $\lambda = 1$ for $x$ from $0$ to $10$ subject to $J(0) = 1$ and $T(10) = 0$. Initial condition is $T(x, t = 0) = 0$.





### Deriving the Weak Form

To transform the strong form of the PDE into the weak form, we follow these standard steps:

1. **Multiply by a test function** $v(x)$:
   $$ v \frac{\partial T}{\partial t} = v \nabla \cdot (\lambda \nabla T) $$

2. **Integrate over the domain** $\Omega = [0, 10]$:
   $$ \int_0^{10} v \frac{\partial T}{\partial t} \, \mathrm{d}x = \int_0^{10} v \frac{\partial}{\partial x} \left( \lambda \frac{\partial T}{\partial x} \right) \, \mathrm{d}x $$

3. **Apply integration by parts** to the right-hand side (diffusion term):
   $$ \int_0^{10} v \frac{\partial}{\partial x} \left( \lambda \frac{\partial T}{\partial x} \right) \, \mathrm{d}x = \left[ v \lambda \frac{\partial T}{\partial x} \right]_0^{10} - \int_0^{10} \frac{\partial v}{\partial x} \lambda \frac{\partial T}{\partial x} \, \mathrm{d}x $$

4. **Substitute back and rearrange**:
   $$ \int_0^{10} v \frac{\partial T}{\partial t} \, \mathrm{d}x + \int_0^{10} \lambda \frac{\partial v}{\partial x} \frac{\partial T}{\partial x} \, \mathrm{d}x = \left[ v \lambda \frac{\partial T}{\partial x} \right]_0^{10} $$

5. **Apply Boundary Conditions**:
   - At $x = 10$, we have a Dirichlet boundary condition $T(10) = 0$. The test function must vanish at Dirichlet boundaries, so $v(10) = 0$.
   - At $x = 0$, we have a Neumann/flux boundary condition $J(0) = 1$. Since flux $J = -\lambda \frac{\partial T}{\partial x}$, we have $-\lambda \frac{\partial T}{\partial x}\Big|_{x=0} = 1$, which means $\lambda \frac{\partial T}{\partial x}\Big|_{x=0} = -1$.
   
   Evaluating the boundary term:
   $$ \left[ v \lambda \frac{\partial T}{\partial x} \right]_0^{10} = v(10) \lambda \frac{\partial T}{\partial x}(10) - v(0) \lambda \frac{\partial T}{\partial x}(0) = 0 - v(0)(-1) = v(0) $$

**Final Weak Form**:
Find $T$ such that for all valid test functions $v$:
$$
\begin{align}
\int_0^{10} v \frac{\partial T}{\partial t} \, \mathrm{d}x + \int_0^{10} \lambda \frac{\partial v}{\partial x} \frac{\partial T}{\partial x} \, \mathrm{d}x = v(0)  \\
\int_0^{10} v \frac{\partial T}{\partial t} \, \mathrm{d}x + \int_0^{10} \lambda v' T' \, \mathrm{d}x = v(0)
\end{align}
$$

6. **Introduce our time integration scheme**:
Now we apply our time stepper. For any given time step $\Delta t$,

$$ \frac{\partial T}{\partial t} \approx \frac{T-T^0}{\Delta t} $$

to get:
$$
\int_0^{10} v T \, \mathrm{d}x - \int_0^{10} v T^0 \, \mathrm{d}x  + \int_0^{10} \lambda v' T' \Delta t \, \mathrm{d}x = v(0) \Delta t
$$


7. **Apply the Galerkin method**:
By substituting the Galerkin approximation $T \approx \sum_j u_j \phi_j(x)$ (where $u_j$ are the unknown nodal temperatures at the current time step, and $u_j^0$ at the previous) and choosing the test function $v$ to be each of the basis functions $\phi_i(x)$, we obtain a system of algebraic equations:

$$
 \underbrace{ \sum_j \left( \int_0^{10} \phi_i \phi_j \, \mathrm{d}x \right)}_{M_{ij}} u_j - \underbrace{\sum_j  \left( \int_0^{10} \phi_i \phi_j \, \mathrm{d}x \right)}_{M_{ij}} u_j^0 + \Delta t \underbrace{\sum_j \left( \int_0^{10} \lambda \phi_i' \phi_j' \, \mathrm{d}x \right)}_{K_{ij}} u_j = \phi_i(0) \Delta t
$$

This naturally defines our **Mass Matrix** ($\mathbf{M}$) and **Stiffness Matrix** ($\mathbf{K}$):
- **Mass Matrix**: $M_{ij} = \int_0^{10} \phi_i \phi_j \, \mathrm{d}x$
- **Stiffness Matrix**: $K_{ij} = \int_0^{10} \lambda \phi_i' \phi_j' \, \mathrm{d}x$

Substituting these matrix definitions, the continuous PDE is reduced to the discrete linear system:
$$ (\mathbf{M} + \Delta t \mathbf{K}) \mathbf{u} = \mathbf{M} \mathbf{u}^0 + \mathbf{f} \Delta t $$
where $\mathbf{f}$ is the boundary load vector with $f_i = \phi_i(0)$.

8. **Apply the element choice**: Only now do we choose our element and apply it. Using Lagrange degree 1 (linear) elements, we define local shape functions for each element interval between local nodes $x_1$ and $x_2$ with $h=x_2-x_1$:

$$\begin{aligned}
\phi_1(x) &= \frac{x_2 - x}{h} \\
\phi_2(x) &= \frac{x - x_1}{h}
\end{aligned}$$

Their spatial derivatives are constants within the element:
$$\begin{aligned}
\nabla \phi_1(x) &= -\frac{1}{h} \\
\nabla \phi_2(x) &= \frac{1}{h}
\end{aligned}$$

9. **Assemble the matrices**: Now we need to assemble the matrices. Generally, this is done with Gauss Points (obtained in the element determined by the element type), but for this example we will do it analytically.

**Element Mass Matrix ($\mathbf{M}_e$)**:
We evaluate the integral $M_{ij} = \int_{x_1}^{x_2} \phi_i(x) \phi_j(x) \, \mathrm{d}x$. For the diagonal and off-diagonal terms, we get:
$$ M_{11} = M_{22} = \int_{x_1}^{x_2} \left(\frac{x_2 - x}{h}\right)^2 \, \mathrm{d}x = \frac{h}{3} $$
$$ M_{12} = M_{21} = \int_{x_1}^{x_2} \left(\frac{x_2 - x}{h}\right) \left(\frac{x - x_1}{h}\right) \, \mathrm{d}x = \frac{h}{6} $$

Factoring out $\frac{h}{6}$ yields the local mass matrix:
$$\mathbf{M}_e = \frac{h}{6} \begin{bmatrix}
2 & 1 \\
1 & 2
\end{bmatrix}$$

**Element Stiffness Matrix ($\mathbf{K}_e$)**:
Similarly, we evaluate the diffusion term $K_{ij} = \lambda \int_{x_1}^{x_2} \nabla \phi_i(x) \cdot \nabla \phi_j(x) \, \mathrm{d}x$:
$$ K_{11} = K_{22} = \lambda \int_{x_1}^{x_2} \left(\pm\frac{1}{h}\right)^2 \, \mathrm{d}x = \frac{\lambda}{h} $$
$$ K_{12} = K_{21} = \lambda \int_{x_1}^{x_2} \left(-\frac{1}{h}\right) \left(\frac{1}{h}\right) \, \mathrm{d}x = -\frac{\lambda}{h} $$

Yielding the local stiffness matrix:
$$\mathbf{K}_e = \frac{\lambda}{h} \begin{bmatrix}
1 & -1 \\
-1 & 1
\end{bmatrix}$$

10. **Global Assembly**:
These matrix *blocks* belong to each individual element. To build the full continuous global matrix, we map the local node numbers to global node numbers and *sum them up*.

For a small mesh with 4 global nodes ($x_1, x_2, x_3, x_4$) connected by 3 elements, the internal nodes are shared. The global matrices are assembled by overlapping the components at the shared vertices, resulting in tridiagonal matrices:

**Global Mass Matrix ($\mathbf{M}$)**:
$$ \mathbf{M} = \begin{bmatrix}
M^{(1)}_{11} & M^{(1)}_{12} & 0 & 0 \\
M^{(1)}_{21} & M^{(1)}_{22} + M^{(2)}_{11} & M^{(2)}_{12} & 0 \\
0 & M^{(2)}_{21} & M^{(2)}_{22} + M^{(3)}_{11} & M^{(3)}_{12} \\
0 & 0 & M^{(3)}_{21} & M^{(3)}_{22}
\end{bmatrix} = \frac{h}{6} \begin{bmatrix}
2 & 1 & 0 & 0 \\
1 & 2+2 & 1 & 0 \\
0 & 1 & 2+2 & 1 \\
0 & 0 & 1 & 2
\end{bmatrix} = \frac{h}{6} \begin{bmatrix}
2 & 1 & 0 & 0 \\
1 & 4 & 1 & 0 \\
0 & 1 & 4 & 1 \\
0 & 0 & 1 & 2
\end{bmatrix} $$

**Global Stiffness Matrix ($\mathbf{K}$)**:
$$ \mathbf{K} = \begin{bmatrix}
K^{(1)}_{11} & K^{(1)}_{12} & 0 & 0 \\
K^{(1)}_{21} & K^{(1)}_{22} + K^{(2)}_{11} & K^{(2)}_{12} & 0 \\
0 & K^{(2)}_{21} & K^{(2)}_{22} + K^{(3)}_{11} & K^{(3)}_{12} \\
0 & 0 & K^{(3)}_{21} & K^{(3)}_{22}
\end{bmatrix} = \frac{\lambda}{h} \begin{bmatrix}
1 & -1 & 0 & 0 \\
-1 & 1+1 & -1 & 0 \\
0 & -1 & 1+1 & -1 \\
0 & 0 & -1 & 1
\end{bmatrix} = \frac{\lambda}{h} \begin{bmatrix}
1 & -1 & 0 & 0 \\
-1 & 2 & -1 & 0 \\
0 & -1 & 2 & -1 \\
0 & 0 & -1 & 1
\end{bmatrix} $$

Note that this overlapping sum at common vertices is exactly how the discrete elements couple together to transfer heat across the entire domain!

```{literalinclude} ../../executable_engineering/executable_engineering/fem_module.py
:language: python
:start-at: def fem_heat_1d_linear
:end-before: def fem_heat_1d_nonlinear_lagged
```

In [ ]:
exe.display_source(exe.fem_heat_1d_linear)

x_nodes, T, M, K = exe.fem_heat_1d_linear(t_final=10.0)
exe.plot_matrix_sparsity(M, title="Sparsity Pattern of Mass Matrix M").show()
exe.plot_fem_heat_1d(mode="linear").show()

## Nonlinear case

Note that in the linear example above we built the $\mathbf{M}$ and $\mathbf{K}$ matrices once, outside the time loop, and reused them for each time step! Even if we had adaptive time stepping, the matrices wouldn't change!

If the PDE is nonlinear, we simply have to use a root finder. Note that in this case we are looking for the parameters $u_i$ directly — the shape functions are unaffected and therefore only the coefficients of the matrix *blocks* are altered. This brings up the concept of *sparsity patterns* which can help reduce the overhead of sparse matrix representations.

Since modern finite element software needs to construct the weak form, it is usually equipped with symbolic logic capabilities and therefore able to calculate the Jacobian for use in the root-finding algorithm.

## Example: 1D nonlinear heat equation

Repeat the above exercise with $\lambda = 1 + T / 10$.

```{literalinclude} ../../executable_engineering/executable_engineering/fem_module.py
:language: python
:start-at: def fem_heat_1d_nonlinear_lagged
:end-before: def fem_heat_1d_nonlinear_root
```

In [ ]:
exe.display_source(exe.fem_heat_1d_nonlinear_lagged)
exe.plot_fem_heat_1d(mode="nonlinear_lagged").show()

## Nonlinear solve using a root finder

Hmmmm... something is funny in that code! Notice that above, $\lambda(T)$ was evaluated explicitly at the *previous* time step (and $\mathbf{M}$ was reassembled every step) rather than solving the coupled nonlinear equation implicitly.

Let's solve the nonlinear system properly at each time step using a root finder:

```{literalinclude} ../../executable_engineering/executable_engineering/fem_module.py
:language: python
:start-at: def fem_heat_1d_nonlinear_root
:end-before: def plot_matrix_sparsity
```

In [ ]:
exe.display_source(exe.fem_heat_1d_nonlinear_root)
exe.plot_fem_heat_1d(mode="nonlinear_root").show()